In [1]:
import spacy
from gliner2 import AutoExtractor

\\bluecrest.local\cfs\Home\angel.adeoye\Projects\dev\financial_research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
CUSTOM_TEXT = (
    "The start of the week has brought more dollar strength. Firmer oil prices, following increased "
    "military activity in the Gulf, have helped keep the dollar on solid footing. Today’s calendar is "
    "unlikely to be particularly impactful, with the NFIB survey and home sales figures the main "
    "highlights. "
    "That said, several technical indicators point to an increasingly stretched USD rally. Our shortterm "
    "valuation model shows the dollar overvalued against every G10 currency except GBP, "
    "where it’s at fair value. On an equally weighted basis, USD is 1.1% overvalued versus G10. EUR, "
    "SEK and NZD stand out as particularly cheap, undershooting fair value by 1.5-2.5% and sitting "
    "beyond the 1.5 standard deviation threshold."
)

In [3]:
model = AutoExtractor.from_pretrained("fastino/gliner2.5-base-v1")

You are using a model of type gliner2 to instantiate a model of type extractor. This is not supported for all configurations of models and can yield errors.
\\bluecrest.local\cfs\Home\angel.adeoye\Projects\dev\financial_research\.venv\Lib\site-packages\gliner2\models\boundary\model.py:2212: UserWarning: Checkpoint uses legacy list-valued extra_special_tokens metadata; loading with compatibility normalization.
  tokenizer = load_extractor_tokenizer(tokenizer_source)


In [4]:
semantic_schema = (
    model.create_schema()

    # TASK 1: SEMANTIC ROLE
    .classification(
        task="semantic_role",
        labels={
            "PRIMARY_THESIS": (
                "A central analytical conclusion, investment thesis, "
                "or outlook that the author advances about a financial, "
                "economic, market, or policy subject. Expresses the "
                "author's main interpretation or conviction."
            ),
            "SUPPORTING_EVIDENCE": (
                "A factual observation, analytical finding, valuation, "
                "positioning measure, or causal explanation used to "
                "support or explain a broader research conclusion. "
                "May itself contain an independently meaningful claim."
            ),
            "RISK_SCENARIO": (
                "An alternative, hypothetical, or conditional outcome "
                "presented as a potential risk to the central outlook. "
                "Not necessarily the author's expected baseline."
            ),
            "BACKGROUND": (
                "Historical events, market developments, economic data, "
                "calendar information, or contextual facts that provide "
                "background rather than a substantive analytical conclusion."
            ),
        },
        multi_label=False,
    )
)

In [5]:
claim_schema = (
    model.create_schema()
    .classification(
        task="claim_type",
        labels={
            "VIEW": (
                "An author's substantive analytical judgment about "
                "market direction, conditions, drivers, risks, or "
                "investment attractiveness. Does not constitute an "
                "explicit trade recommendation, fair-value assessment, "
                "or specific expected future outcome."
            ),
            "FORECAST": (
                "An expected future financial, economic, market, or "
                "policy outcome endorsed by the author. Includes "
                "directional expectations, numerical targets, and "
                "explicit baseline predictions. Excludes hypothetical "
                "possibilities and forecasts merely attributed to others."
            ),
            "VALUATION": (
                "An assessment of fair value, intrinsic value, "
                "relative value, or mispricing of an asset or instrument. "
                "Includes explicit assessments that something is cheap, "
                "expensive, overvalued, or undervalued. Excludes "
                "historical price movements and technical overbought signals."
            ),
            "POSITIONING": (
                "A statement about actual financial exposures held "
                "by investors or market participants, including "
                "long or short positions, holdings, allocations, "
                "and changes in exposures. Excludes price movements, "
                "market sentiment, and market-implied expectations."
            ),
            "TRADE_IDEA": (
                "An explicit recommendation or proposed trading strategy "
                "to establish, adjust, hedge, or close an exposure "
                "in a financial instrument, market, or asset class. "
                "Requires a recommended action or exposure. "
                "Excludes directional opinions without a trading recommendation."
            ),
            "NONE": (
                "The statement does not express an independently "
                "meaningful financial claim belonging to the five "
                "defined categories. Includes purely descriptive "
                "information and explanations without a qualifying claim."
            ),
        },
        multi_label=False,
    )
)

In [7]:

schema = (
    model.create_schema()
    .classification(
        task="semantic_role",
        labels={
            "PRIMARY_THESIS": (
                "The author's central investment conclusion or outlook."
            ),
            "SUPPORTING_EVIDENCE": (
                "Facts, observations, or causal explanations supporting "
                "a broader financial conclusion."
            ),
            "RISK_SCENARIO": (
                "A conditional, hypothetical, or alternative outcome."
            ),
            "BACKGROUND": (
                "Historical events, descriptive facts, or context."
            ),
        },
        multi_label=False,
    )
    .classification(
        task="claim_eligibility",
        labels={
            "CLAIM": (
                "An independently meaningful financial assertion "
                "expressing an analytical view, an endorsed future "
                "expectation, a valuation, actual financial positioning, "
                "or an explicit trading recommendation."
            ),
            "NO_CLAIM": (
                "A statement that only reports historical price movements, "
                "explains previously observed market activity, provides "
                "context, or reports routine economic data, without "
                "expressing an independent qualifying financial claim."
            ),
        },
        multi_label=False,
    )
    .classification(
        task="claim_type",
        labels={
            "VIEW": (
                "An independent analytical assessment of a financial "
                "subject, including directional conviction or a "
                "material current market judgment. Not merely an "
                "explanation of historical price movements."
            ),
            "FORECAST": (
                "An explicit prediction or expectation of a future "
                "financial, economic, or policy outcome endorsed "
                "by the author. Excludes explanations of past or "
                "present market behaviour."
            ),
            "VALUATION": (
                "An assessment of fair value, relative value, "
                "overvaluation, undervaluation, or mispricing."
            ),
            "POSITIONING": (
                "A statement about actual investor holdings, "
                "allocations, or long and short market exposures."
            ),
            "TRADE_IDEA": (
                "An explicit proposal or recommendation to enter, "
                "modify, hedge, or exit a financial position."
            ),
        },
        multi_label=False,
    )
)


In [10]:
sentence = (
    "Excluding JPY, where long positions remain unusually elevated, net long USD positioning versus G10 reached 24% of open interest as of last Tuesday"
)

sentence = "“We expect higher oil prices to support further USD appreciation”"

results = model.extract(sentence, schema, include_confidence=True)

print("RESULTS:", results)

RESULTS: {'semantic_role': {'label': 'SUPPORTING_EVIDENCE', 'confidence': 0.8510182499885559}, 'claim_eligibility': {'label': 'CLAIM', 'confidence': 0.9999822378158569}, 'claim_type': {'label': 'FORECAST', 'confidence': 0.998042106628418}}


In [ ]:
# Lightweight sentence segmentation; no model download needed
nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")

doc = nlp(CUSTOM_TEXT)

results = []

for i, sent in enumerate(doc.sents):
    sentence = sent.text.strip()

    # Stage 1: Semantic role
    semantic = model.extract(sentence, semantic_schema)
    role = semantic["semantic_role"]

    # Stage 2: Financial claim type
    claim_type = None

    if role == "ASSERTION":
        claim = model.extract(sentence, claim_schema)
        claim_type = claim["claim_type"]

        if claim_type == "NONE":
            claim_type = None

    results.append({
        "sentence_id": i,
        "text": sentence,
        "start": sent.start_char,
        "end": sent.end_char,
        "semantic_role": role,
        "claim_type": claim_type,
    })

for result in results:
    print(f"\n[{result['sentence_id']}] {result['text']}")
    print(f"Semantic: {result['semantic_role']}")
    print(f"Claim: {result['claim_type']}")

In [47]:
from gliner2 import RegexValidator

sentence_validator = RegexValidator(
    pattern=r"(?s)^\s*\S.{8,}[.!?][”\"']?\s*$",
    mode="full",
)

schema = (
    model.create_schema()
    .structure(
        "statement",
        mode="natural",
        anchor="sentence",
    )
    .field(
        "sentence",
        dtype="str",
        cardinality="required_one",
        validators=[sentence_validator],
        description="One complete sentence copied verbatim from the source.",
    )
    .field(
        "semantic_role",
        dtype="str",
        choices=[
            "PRIMARY_THESIS",
            "SUPPORTING_EVIDENCE",
            "RISK_SCENARIO",
            "BACKGROUND",
        ],
        cardinality="required_one",
    )
    .field(
        "claim_type",
        dtype="str",
        choices=[
            "VIEW",
            "FORECAST",
            "VALUATION",
            "POSITIONING",
            "TRADE_IDEA",
            "NONE",
        ],
        cardinality="required_one",
    )
)


In [48]:
result = model.extract(
    CUSTOM_TEXT,
    schema,
    include_confidence=True,
)

from pprint import pprint
pprint(result, width=120)

{}
